In [ ]:
from collections import defaultdict

In [ ]:
def get_all_pairs(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        chrs = word.split()
        for i in range(len(chrs)-1):
            pairs[(chrs[i], chrs[i+1])] += freq
    return pairs


def merge(best_pair, vocab):
    n_gram = " ".join(best_pair)
    replacement = "".join(best_pair)
    new_vocab = {}
    for word, freq in vocab.items():
        new_word = word.replace(n_gram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab 


def train_bpe(corpus, n_merges):
    vocab = defaultdict(int)
    for word in corpus.split():
        vocab[" ".join(word) + " </w>"] += 1
    print(vocab)

    merges = []
    for _ in range(n_merges):
        pairs = get_all_pairs(vocab)
        best_pair = max(pairs, key=pairs.get)
        vocab = merge(best_pair, vocab)
        merges.append(best_pair)
        print(f"Merged {best_pair} => {"".join(best_pair)}")

    return merges, vocab


def tokenize(word, merges):
    tokens = list(word) + ["</w>"]
    for (a, b) in merges:
        new = []
        i = 0
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == a and tokens[i+1]==b:
                new.append(a+b)
                i += 2
            else:
                new.append(tokens[i])
                i += 1
        tokens = new
    return tokens




def tokenize_corpus(corpus, merges):
    tokens = []
    for word in corpus.split():
        tokens.extend(tokenize(word, merges))
    return tokens




In [ ]:
corpus = "The cat is running across the street."

merges, vocab = train_bpe(corpus, 4)
print(merges)

test_corpus = "I am going to the park." 
print(" ".join(tokenize_corpus(test_corpus, merges)))